# Tutorial 1: Design of Experiments

Before running any simulations, we need to choose a set of parameter points
that efficiently covers the prior space. This notebook demonstrates:

1. **Latin Hypercube Sampling (LHS)** — good space-filling for GP training
2. **Sobol sequences** — quasi-random, low-discrepancy alternative
3. Visualising the design with pair plots

> **First time?** Run `python scripts/generate_toy_data.py` from the repo root to create the synthetic data files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations

from hic_bayes import lhs_design, sobol_design

# Parameter bounds for a 5D heavy-ion model
param_bounds = {
    "eta_s":      (0.00, 0.50),
    "zeta_s_max": (0.00, 0.20),
    "tau_0":      (0.10, 1.50),
    "Qs_mu":      (0.50, 2.00),
    "T_switch":   (0.13, 0.18),
}

In [ ]:
lhs = lhs_design(param_bounds, n_points=100, seed=42)
sobol = sobol_design(param_bounds, n_points=128, seed=42)

print(f"LHS design:   {lhs.shape}")
print(f"Sobol design: {sobol.shape}")
lhs.head()

In [ ]:
params = list(param_bounds.keys())
pairs = list(combinations(range(len(params)), 2))
fig, axes = plt.subplots(2, len(pairs)//2 + 1, figsize=(14, 6))
axes = axes.flatten()

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(lhs.iloc[:, i], lhs.iloc[:, j], s=8, alpha=0.7, label='LHS')
    ax.scatter(sobol.iloc[:, i], sobol.iloc[:, j], s=8, alpha=0.4, label='Sobol', marker='s')
    ax.set_xlabel(params[i], fontsize=8)
    ax.set_ylabel(params[j], fontsize=8)

for ax in axes[len(pairs):]:
    ax.set_visible(False)

axes[0].legend(fontsize=8)
plt.suptitle('Design point distributions — pairwise projections', y=1.01)
plt.tight_layout()
plt.show()

## Notes

- **LHS** guarantees one point per bin in each dimension — good for GP training.
- **Sobol** has better higher-dimensional uniformity and is preferred for larger designs (N > 200).
- Rule of thumb: **N ≥ 10 × d** design points for reliable GP emulation in `d` dimensions.